# Session 3, Module 04: Context Managers


This module covers:
- Understanding the 'with' statement
- Creating context managers with __enter__/__exit__
- Using @contextmanager decorator
- Practical patterns for resource management

Data Engineering Context:
Context managers ensure resources (files, connections, locks)
are properly cleaned up, even when errors occur.


In [2]:
from contextlib import contextmanager
import time
import tempfile
from pathlib import Path

## Understanding The 'With' Statement


In [3]:
print("=== Understanding the 'with' Statement ===")

=== Understanding the 'with' Statement ===


The 'with' statement ensures cleanup happens automatically
Even if an exception occurs
Without 'with' — manual cleanup (error-prone)

In [5]:
print("\nWithout 'with' (manual cleanup):")
temp_file = Path(tempfile.gettempdir()) / "test_without_with.txt"
f = open(temp_file, "w")
try:
    f.write("Hello, World!")
finally:
    f.close()
    print("  File closed manually")

# With 'with' — automatic cleanup
print("\nWith 'with' (automatic cleanup):")
with open(temp_file, "w") as f:
    f.write("Hello, World!")
print("  File closed automatically")


Without 'with' (manual cleanup):
  File closed manually

With 'with' (automatic cleanup):
  File closed automatically


The 'with' statement calls:
1. __enter__() at the start
2. __exit__() at the end (even if exception occurs)

## Creating Context Managers With Class


In [9]:
print("\n=== Creating Context Managers with Class ===")


class Timer:
    """Context manager for timing code blocks."""

    def __init__(self, name: str = "Operation"):
        self.name = name
        self.start_time = None
        self.end_time = None

    def __enter__(self):
        """Called when entering the 'with' block."""
        self.start_time = time.perf_counter()
        print(f"  [{self.name}] Starting...")
        return self  # This is what gets assigned to 'as' variable

    def __exit__(self, exc_type, exc_val, exc_tb):
        """
        Called when exiting the 'with' block.

        Args:
            exc_type: Exception type (if any)
            exc_val: Exception value (if any)
            exc_tb: Exception traceback (if any)

        Returns:
            True to suppress exception, False to propagate
        """
        self.end_time = time.perf_counter()
        elapsed = self.end_time - self.start_time
        print(f"  [{self.name}] Completed in {elapsed:.4f} seconds")

        return False


# Usage
with Timer("Data processing") as t:
    data = [x ** 2 for x in range(10000)]
    time.sleep(0.1)
    


=== Creating Context Managers with Class ===
  [Data processing] Starting...
  [Data processing] Completed in 0.1059 seconds


Return False to propagate any exception
Return True to suppress the exception

## Context Manager For Database Connections


In [10]:
print("\n=== Context Manager for Database Connections ===")


class DatabaseConnection:
    """
    Simulated database connection with context manager.

    Ensures connection is always closed, even on errors.
    """

    def __init__(self, host: str, database: str):
        self.host = host
        self.database = database
        self.connected = False
        self.queries_executed = 0

    def __enter__(self):
        """Establish connection."""
        print(f"  Connecting to {self.database}@{self.host}...")
        self.connected = True
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        """Close connection."""
        if exc_type:
            print(f"  Error occurred: {exc_val}")
            print("  Rolling back transaction...")
        else:
            print("  Committing transaction...")

        print(f"  Closing connection (executed {self.queries_executed} queries)")
        self.connected = False
        return False  # Don't suppress exceptions

    def execute(self, query: str) -> list:
        """Execute a query."""
        if not self.connected:
            raise RuntimeError("Not connected to database")
        print(f"  Executing: {query[:50]}...")
        self.queries_executed += 1
        return [{"result": "data"}]


# Normal usage
print("\nSuccessful database operation:")
with DatabaseConnection("localhost", "warehouse") as db:
    db.execute("SELECT * FROM customers")
    db.execute("SELECT * FROM orders")

# Usage with error
print("\nDatabase operation with error:")
try:
    with DatabaseConnection("localhost", "warehouse") as db:
        db.execute("SELECT * FROM customers")
        raise ValueError("Simulated error")
except ValueError:
    print("  Exception was propagated (not suppressed)")


=== Context Manager for Database Connections ===

Successful database operation:
  Connecting to warehouse@localhost...
  Executing: SELECT * FROM customers...
  Executing: SELECT * FROM orders...
  Committing transaction...
  Closing connection (executed 2 queries)

Database operation with error:
  Connecting to warehouse@localhost...
  Executing: SELECT * FROM customers...
  Error occurred: Simulated error
  Rolling back transaction...
  Closing connection (executed 1 queries)
  Exception was propagated (not suppressed)


## Using @Contextmanager Decorator


In [11]:
print("\n=== Using @contextmanager Decorator ===")


=== Using @contextmanager Decorator ===


The @contextmanager decorator simplifies creating context managers
using generator syntax

In [12]:
@contextmanager
def timer(name: str = "Operation"):
    """
    Context manager for timing code blocks.

    Usage:
        with timer("My operation"):
            do_something()
    """
    start = time.perf_counter()
    print(f"  [{name}] Starting...")

    try:
        yield  # Everything before yield is __enter__, after is __exit__
    finally:
        elapsed = time.perf_counter() - start
        print(f"  [{name}] Completed in {elapsed:.4f} seconds")


# Usage
with timer("Quick calculation"):
    result = sum(range(100000))


# More complex example with yielded value
@contextmanager
def temp_directory(prefix: str = "temp"):
    """
    Create a temporary directory that is cleaned up after use.

    Yields:
        Path to the temporary directory
    """
    import shutil

    temp_dir = Path(tempfile.mkdtemp(prefix=f"{prefix}_"))
    print(f"  Created temp directory: {temp_dir}")

    try:
        yield temp_dir
    finally:
        print(f"  Cleaning up temp directory: {temp_dir}")
        shutil.rmtree(temp_dir, ignore_errors=True)


# Usage
print("\nTemporary directory context manager:")
with temp_directory("etl_job") as tmp:
    # Create some files
    (tmp / "data.txt").write_text("temporary data")
    print(f"  Created file in {tmp}")
# Directory is automatically cleaned up

  [Quick calculation] Starting...
  [Quick calculation] Completed in 0.0015 seconds

Temporary directory context manager:
  Created temp directory: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/etl_job_ik6jd3j7
  Created file in /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/etl_job_ik6jd3j7
  Cleaning up temp directory: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/etl_job_ik6jd3j7


## Context Manager For Transaction


In [13]:
print("\n=== Context Manager for Transaction ===")


@contextmanager
def transaction(name: str):
    """
    Simulated transaction context manager.

    Commits on success, rolls back on error.
    """
    print(f"  BEGIN TRANSACTION: {name}")

    try:
        yield
        print(f"  COMMIT: {name}")
    except Exception as e:
        print(f"  ROLLBACK: {name} (Error: {e})")
        raise


# Successful transaction
print("\nSuccessful transaction:")
with transaction("Update customers"):
    print("  Updating records...")

# Failed transaction
print("\nFailed transaction:")
try:
    with transaction("Update orders"):
        print("  Updating records...")
        raise RuntimeError("Constraint violation")
except RuntimeError:
    pass


=== Context Manager for Transaction ===

Successful transaction:
  BEGIN TRANSACTION: Update customers
  Updating records...
  COMMIT: Update customers

Failed transaction:
  BEGIN TRANSACTION: Update orders
  Updating records...
  ROLLBACK: Update orders (Error: Constraint violation)


## Nested Context Managers


In [14]:
print("\n=== Nested Context Managers ===")


# Multiple context managers can be nested
@contextmanager
def log_context(name: str):
    """Add context to log messages."""
    print(f"  [ENTER {name}]")
    try:
        yield
    finally:
        print(f"  [EXIT {name}]")


# Nested usage
print("\nNested context managers:")
with log_context("outer"):
    with log_context("inner"):
        print("  Doing work...")

# Multiple on same line (Python 3.10+)
print("\nMultiple context managers (single line):")
with log_context("A"), log_context("B"):
    print("  Doing work...")


=== Nested Context Managers ===

Nested context managers:
  [ENTER outer]
  [ENTER inner]
  Doing work...
  [EXIT inner]
  [EXIT outer]

Multiple context managers (single line):
  [ENTER A]
  [ENTER B]
  Doing work...
  [EXIT B]
  [EXIT A]


## Practical Example: File Backup


In [15]:
print("\n=== Practical Example: File Backup ===")


@contextmanager
def backup_file(file_path: Path):
    """
    Create a backup before modifying a file.

    Restores backup if an error occurs.
    """
    backup_path = file_path.with_suffix(file_path.suffix + ".bak")

    # Create backup if file exists
    if file_path.exists():
        import shutil
        shutil.copy2(file_path, backup_path)
        print(f"  Created backup: {backup_path}")

    try:
        yield file_path
        # Success - remove backup
        if backup_path.exists():
            backup_path.unlink()
            print(f"  Removed backup (success)")
    except Exception:
        # Error - restore backup
        if backup_path.exists():
            import shutil
            shutil.move(backup_path, file_path)
            print(f"  Restored from backup (error occurred)")
        raise


# Demonstrate backup
temp_file = Path(tempfile.gettempdir()) / "important_data.txt"
temp_file.write_text("Original content")

print("\nSuccessful modification:")
with backup_file(temp_file) as f:
    f.write_text("Modified content")
print(f"  Final content: {temp_file.read_text()}")

# Reset file
temp_file.write_text("Original content again")

print("\nFailed modification (backup restored):")
try:
    with backup_file(temp_file) as f:
        f.write_text("Bad modification")
        raise ValueError("Something went wrong!")
except ValueError:
    pass
print(f"  Final content: {temp_file.read_text()}")


=== Practical Example: File Backup ===

Successful modification:
  Created backup: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/important_data.txt.bak
  Removed backup (success)
  Final content: Modified content

Failed modification (backup restored):
  Created backup: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/important_data.txt.bak
  Restored from backup (error occurred)
  Final content: Original content again


## Context Manager For Changing State


In [16]:
print("\n=== Context Manager for Changing State ===")


@contextmanager
def set_env_var(name: str, value: str):
    """
    Temporarily set an environment variable.

    Restores original value (or unsets) after the block.
    """
    import os

    old_value = os.environ.get(name)
    os.environ[name] = value
    print(f"  Set {name}={value}")

    try:
        yield
    finally:
        if old_value is None:
            del os.environ[name]
            print(f"  Unset {name}")
        else:
            os.environ[name] = old_value
            print(f"  Restored {name}={old_value}")


# Usage
import os
print("\nTemporary environment variable:")
print(f"  Before: MY_VAR = {os.environ.get('MY_VAR', 'NOT SET')}")

with set_env_var("MY_VAR", "temporary_value"):
    print(f"  Inside: MY_VAR = {os.environ.get('MY_VAR')}")

print(f"  After: MY_VAR = {os.environ.get('MY_VAR', 'NOT SET')}")


=== Context Manager for Changing State ===

Temporary environment variable:
  Before: MY_VAR = NOT SET
  Set MY_VAR=temporary_value
  Inside: MY_VAR = temporary_value
  Unset MY_VAR
  After: MY_VAR = NOT SET


## Context Manager For Locking


In [17]:
print("\n=== Context Manager for Locking ===")


class FileLock:
    """
    Simple file-based lock for cross-process synchronization.

    Usage:
        with FileLock("/tmp/my_job.lock"):
            # Only one process can run this code at a time
            process_data()
    """

    def __init__(self, lock_path: str, timeout: float = 10.0):
        self.lock_path = Path(lock_path)
        self.timeout = timeout
        self.acquired = False

    def __enter__(self):
        start_time = time.time()

        while self.lock_path.exists():
            if time.time() - start_time > self.timeout:
                raise TimeoutError(f"Could not acquire lock: {self.lock_path}")
            print("  Waiting for lock...")
            time.sleep(0.1)

        self.lock_path.write_text(str(time.time()))
        self.acquired = True
        print(f"  Lock acquired: {self.lock_path}")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.acquired and self.lock_path.exists():
            self.lock_path.unlink()
            print(f"  Lock released: {self.lock_path}")
        return False


# Usage
lock_file = Path(tempfile.gettempdir()) / "pipeline.lock"
with FileLock(str(lock_file)):
    print("  Processing with lock held...")
    time.sleep(0.1)


=== Context Manager for Locking ===
  Lock acquired: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/pipeline.lock
  Processing with lock held...
  Lock released: /var/folders/jm/62gxxtv50mv5df3587g4w99r0000gn/T/pipeline.lock


## Summary


In [18]:
print("\n=== Summary ===")
print("""
Context Managers Key Points:

WITH STATEMENT:
  - Ensures cleanup happens automatically
  - Works even when exceptions occur
  - Calls __enter__ and __exit__

CLASS-BASED:
  class MyManager:
      def __enter__(self):
          # Setup
          return self
      def __exit__(self, exc_type, exc_val, exc_tb):
          # Cleanup
          return False  # Propagate exceptions

@contextmanager DECORATOR:
  @contextmanager
  def my_manager():
      # Setup (before yield)
      try:
          yield resource  # Value available via 'as'
      finally:
          # Cleanup (always runs)

COMMON PATTERNS:
  - Database connections (connect/disconnect)
  - File handling (open/close)
  - Locks (acquire/release)
  - Transactions (begin/commit or rollback)
  - Temporary state changes (set/restore)
  - Timing blocks (start/stop timer)
""")


=== Summary ===

Context Managers Key Points:

WITH STATEMENT:
  - Ensures cleanup happens automatically
  - Works even when exceptions occur
  - Calls __enter__ and __exit__

CLASS-BASED:
  class MyManager:
      def __enter__(self):
          # Setup
          return self
      def __exit__(self, exc_type, exc_val, exc_tb):
          # Cleanup
          return False  # Propagate exceptions

@contextmanager DECORATOR:
  @contextmanager
  def my_manager():
      # Setup (before yield)
      try:
          yield resource  # Value available via 'as'
      finally:
          # Cleanup (always runs)

COMMON PATTERNS:
  - Database connections (connect/disconnect)
  - File handling (open/close)
  - Locks (acquire/release)
  - Transactions (begin/commit or rollback)
  - Temporary state changes (set/restore)
  - Timing blocks (start/stop timer)

